# 03 · Annotate, adjudicate → *your* gold set

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/egumasa/lda2-final-template/blob/main/notebooks/03_annotate.ipynb)

The part no model can do for you, and the part the Q&A will ask about.

```
  01_build_pool_<track>  →  02_sample  →▶ 03_annotate  →  04_prompt  →  05_report
```

| | |
|---|---|
| **Reads** | `data/gold/<track>_<group>_sample.json` (from 02) |
| **Writes** | `data/gold/<track>_<group>_gold.json` |

---

The published labels are somebody else's judgment. Before you measure a model against them, two of you re-annotate the sample **independently and blind**, see how far apart you were, and argue out the rows you disagreed on.

What comes out is *your* gold set — and the disagreements tell you which label boundaries are genuinely fuzzy. That is what lets you say, later, whether a model's miss is the **model's** fault or the **scheme's**. Nothing else in the project can tell you that.

> Budget real time for this. Forty items, two annotators, plus the argument afterwards. It is the most valuable thing you will make this week and the easiest to rush.

## Setup — run this first

In Colab, uncomment **one** of the two clone blocks below before running. Colab starts with only this one file; the clone fetches everything *around* it (`scripts/`, `config.py`, `data/`) so the paths resolve.

**Do Option A once, as a group** — then always open the copy in Drive (*File ▸ Open ▸ Drive ▸ `lda2-final-template/notebooks/...`*). Your prompts, gold set and outputs then survive the runtime resetting, and everyone sees the same files.

In [ ]:
# ------------------------------------------------------------------
# SETUP — run me first.
# ------------------------------------------------------------------
# In Google Colab, UNCOMMENT one of the two blocks below, then run the cell.

# --- Colab Option A: clone into your Google Drive (persists; do this once) ---
# from google.colab import drive
# drive.mount("/content/drive")
# %cd /content/drive/MyDrive
# ![ -d lda2-final-template ] || git clone https://github.com/egumasa/lda2-final-template.git lda2-final-template
# %cd /content/drive/MyDrive/lda2-final-template/notebooks

# --- Colab Option B: quick, throwaway clone (changes lost on reset) ---
# !git clone https://github.com/egumasa/lda2-final-template.git
# %cd lda2-final-template/notebooks

# Put scripts/ and config.py on the import path. Works locally AND in Colab
# after the %cd above, because notebooks/ sits beside both.
import sys
sys.path.append("../scripts")
sys.path.append("..")

from config import *      # TRACK, GROUP, SEED, N_PER_CLASS, and every path

from pipeline import *      # load_gold, label_set, save_json, ...
from annotate import *      # create_annotation_sheet, annotator_agreement, ...

describe()                  # what this notebook is working on


> **Everything above comes from `config.py`** — one file at the top of the repo, which you edit once as a group. That is deliberate: the seed in notebook 02 has to be the seed in notebook 03, and five copies of a number in five notebooks is five chances for them to disagree. If the line it just printed is not your track, your group and your seed, fix `config.py` and re-run this cell.

In [ ]:
# ══ STEP 1 · Load your sample ═════════════════════════════════════════════
# Goal      : open the forty items notebook 02 drew.
# Available : load_gold(path)  ->  a list of {id, text, label}
#             SAMPLE_PATH   (from config.py)
# Pointer   : Day 2 S5 step F — the same call.
# Produce   : sampled · LABELS      ← later cells use these names
# Note      : it still carries the published label. Do not read it, and do
#             not print it — you are about to annotate these blind.

# ✏️ your code here


## Step 2 — Create the annotation sheet

A real Google Sheet in your own Drive, one row per item, with blank `CoderA`, `CoderB`, `Final` and `Note` columns. All of you can have it open at once.

The published label is **deliberately not copied in**. Two people annotate independently, without seeing each other's column or the corpus's answer — that is what makes the agreement number mean anything. Decide who is CoderA and who is CoderB before you start, and do not look across.

**On `cars50` and `raamove`** the sheet gets one extra column, `Context`: the passage each sentence came from, with the sentence you are labelling marked `>>>`. Read it. A move is a rhetorical function *within* a passage, and if the sentence alone is too thin for the model to judge, it is just as thin for the two of you — and your labels are the answer key everything else gets measured against.

The first time you run this, Colab asks for permission to use your Google account. That is `gspread` authorising against your own Drive.

In [ ]:
# ══ STEP 2 · Create the sheet ═════════════════════════════════════════════
# Goal      : make a blind annotation sheet in your Drive, one row per sampled item.
# Available : create_annotation_sheet(title, items, labels)  ->  the sheet URL
# Pointer   : Day 2 S5 step A — the same call.
# Produce   : a sheet URL — paste it into SHEET_ID below      ← later cells use these names
# Note      : give it a title with your group and track in it. You will
#             have several of these by the end of the week.
# Careful   : run this ONCE. Running it again makes a SECOND sheet, and
#             half your annotations end up in the one nobody read back.

# ✏️ your code here


### Now go and annotate

Open the sheet, and label every row. Rules of the exercise:

- **CoderA and CoderB work independently.** Different people, no discussion, no peeking at the other column.
- **Leave `Final` empty** until you have both finished and talked.
- **Use `Note`** when you hesitate. The item you were unsure about is the item you will want to quote in your error analysis, and you will not remember which one it was.
- Labels must be spelled exactly as `LABELS` prints them. `to_canonical` will tell you about typos, but it is quicker not to make them.

This is the point where the notebook stops and the week's actual work happens. Come back when both columns are full.

In [ ]:
# Paste the URL your sheet printed above (or just the long id from it).
# It lives here rather than in config.py because it is per-round, not per-group.
SHEET_ID = ""
ROUND    = "round1"          # each re-annotation round gets its own tab


## Step 3 — Measure agreement

Two numbers and a matrix: raw percent agreement, Cohen's κ (agreement corrected for what you would get by guessing), and an annotator-vs-annotator confusion matrix whose off-diagonal cells show *which* label pairs the two of you confuse.

**Write these down now** — they are report section 1, and they do not survive a runtime reset. A κ around .8 is strong; around .4 means the scheme, not the annotators, is doing something wrong. Either is a reportable finding. A low κ you can explain beats a high one you cannot.

In [ ]:
# ══ STEP 3 · Measure agreement ════════════════════════════════════════════
# Goal      : how often did the two of you agree, and on which labels did you not?
# Available : load_annotation_sheet(sheet_id, worksheet)  ->  rows
#             annotator_agreement(rows)  ·  disagreements(rows)
# Pointer   : Day 2 S5 steps D–E — identical calls.
# Produce   : rows · disagreed      ← later cells use these names
# Note      : run this once BOTH CoderA and CoderB columns are filled in.
#             Half-finished rows are dropped from the comparison.
# Keep      : `disagreed` matters again in notebook 05 — the rows you
#             argued about are the ones to check your model's errors
#             against. Save the list, or write the ids down.

# ✏️ your code here


## Step 4 — Adjudicate

Go back to the sheet and fill in `Final` for **every** row:

- Where you agreed, `Final` is that label.
- Where you did not, talk it out and decide. If you cannot agree, the scheme is underspecified — write down *why* in `Note` and pick one. That note is worth more to your report than the label is.

Then re-read the sheet and canonicalise it. `to_canonical` reports blanks and invalid labels rather than silently dropping them; fix them in the sheet and re-run until it says **0 blank, 0 invalid**.

In [ ]:
# ══ STEP 4 · Adjudicate, then canonicalise ════════════════════════════════
# Goal      : agree a Final label for every row, then turn the sheet into gold.
# Available : load_annotation_sheet(sheet_id, worksheet)  ->  rows   (re-read after editing)
#             to_canonical(rows, LABELS, source=sampled)  ->  gold
# Pointer   : Day 2 S5 step F — identical calls.
# Produce   : gold      ← later cells use these names
# Careful   : re-read the sheet first. `rows` from step 3 is a snapshot
#             from before you filled in Final.
# Note      : keep going until it prints 0 blank and 0 invalid. A blank
#             row is an item silently missing from your study.
# Note      : pass source=sampled. Gold is rebuilt from the SHEET, which
#             holds only the id, the text and your label — anything else
#             the item carried (on cars50/raamove, its passage) is put
#             back from `sampled` by id. Harmless on the other tracks.

# ✏️ your code here


## Step 5 — Where do you differ from the published labels?

Now — and only now, with your own labels settled — look at what the corpus said. `compare_to_published` matches by text and shows you every row where your group landed somewhere else.

**Disagreement here is not an error.** You annotated forty items carefully against a scheme you had thought about; the original annotators worked at scale under different guidelines. Where you differ, one of three things is true, and saying which is exactly the analytical work this project is for:

1. **Your scheme drifted** from theirs — you read a category boundary differently. Say where.
2. **The item is genuinely ambiguous** — it would split any pair of annotators.
3. **One of you is wrong.** It happens, in both directions.

This table is report section 1, and it is the one that most often produces a sentence worth saying out loud in the Q&A.

In [ ]:
# ══ STEP 5 · Compare against the published labels ═════════════════════════
# Goal      : see where your gold set and the corpus disagree, and work out why.
# Available : compare_to_published(gold, sampled)  ->  a table of the rows that differ
# Pointer   : Day 2 S5 step F — the same call.
# Produce   : differences      ← later cells use these names
# Careful   : compare against `sampled`, not `pool`. Sampling renumbers
#             the ids, so `pool` would line your item 7 up against a
#             different sentence entirely.
# Note      : pick two or three and write down which of the three cases
#             above they are. Do it now, while you remember the argument.

# ✏️ your code here


## Save it — this is the handoff

This file is the single most valuable thing your group makes all week — hours of judgment, and the only thing in the project that could not have been produced by a script. Every number in notebooks 04 and 05 is measured against it, and it goes in your submission bundle.

**Next:** open `04_prompt.ipynb`. It starts by loading `data/gold/<track>_<group>_gold.json`.

In [ ]:
save_json(gold, GOLD_PATH, what="gold items")

# It is git-ignored — it is your work, not part of the template. If you cloned
# into Google Drive it is already saved across sessions; if not, download it.


---

## 🛑 The `PLAN.md` gate

Notebook 04 starts calling the model. **Do not open it until your `PLAN.md` has been read and signed off.** It takes two minutes and it is not busywork: a mismatched label set or an unstated sampling seed costs an hour to unpick *after* you have burned quota on it.

Check, out loud, that these three agree: the label set in `PLAN.md`, the labels `label_set` actually returned above, and the labels your prompt file names.